In [1]:
import itertools
from math import comb

import numpy as np
import qutip as qt

def all_codewords(N, b, K_min=2):
    """
    Every valid codeword |v_S> = (|S> + |S^c>)/sqrt(2) of the imbalance code in sector b.

    Valid means:
        - each mirror pair p = (p, N-1-p) is empty or active (one excitation); no doubles
        - b even, so M = 0 puts b/2 excitations on each half
        - K = b/2 >= K_min, so dim C = C(2K, K)/2 >= 3
        - b <= N/2, at most one excitation per pair
        - S and S^c are the same codeword; keep the label containing the first active pair

    Yields:
        t     : pair assignment, tuple of 0/1 of length N/2
        S     : frozenset of active pairs whose excitation is on the LEFT site
        occS  : sorted occupied sites of |S>
        occSc : sorted occupied sites of |S^c> (the mirror image)
    """
    m = N // 2
    if N % 2 or b % 2 or b < 2 * K_min or b > m:
        return
    K = b // 2

    for A in itertools.combinations(range(m), b):          # active pairs; all others empty
        t = tuple(1 if p in A else 0 for p in range(m))
        for S in itertools.combinations(A, K):
            if A[0] not in S:                               # drop the S^c duplicate
                continue
            S = frozenset(S)
            occS  = sorted(p if p in S else N - 1 - p for p in A)
            occSc = sorted(N - 1 - p if p in S else p for p in A)
            yield t, S, occS, occSc

def get_V_W(N, alpha, Jz):
    """V(i) = -2 Jz sum_{j!=i} |i-j|^-alpha,  W(i,j) = 4 Jz |i-j|^-alpha (W[i,i] = 0)."""
    r = np.abs(np.subtract.outer(np.arange(N), np.arange(N))).astype(float)
    np.fill_diagonal(r, np.inf)
    W = 4.0 * Jz * r ** (-alpha)
    V = -0.5 * W.sum(axis=1)
    return V, W

def config_energy(occ, V, W):
    """
    Diagonal energy of a configuration,

        E(c) = sum_{i in c} V(i)  +  sum_{i < j in c} W(i, j)

    occ : list of occupied sites, e.g. [4, 5, 6]

    will return V[4]+V[5]+V[6]+W[4,5]+W[4,6]+W[5,6]
    """
    # Turn the list into a numpy array so it can be used to index V and W.
    occ = np.asarray(occ)

    # V[occ] picks out the entries of V at the occupied sites and .sum() adds them: 
    # this is the potential-energy term.
    potential = V[occ].sum()

    # np.ix_(occ, occ) selects the rows AND columns in occ, i.e. the sub-matrix
    # of W restricted to occupied sites. For occ = [4, 5]:
    #     [[W(4,4) W(4,5)]      [[0. 4.]
    #      [W(5,4) W(5,5)]]  =   [4. 0.]]
    # Summing it gives W(4,5) + W(5,4) = 2 W(4,5): every pair is counted twice
    # (once as (i,j), once as (j,i)), and the diagonal is 0.
    # The factor 0.5 removes the double counting -> sum over i < j only.
    repulsion = 0.5 * W[np.ix_(occ, occ)].sum()

    return potential + repulsion

def build_Heff_sector(N, b, alpha, Jz, Jx):
    V, W = get_V_W(N, alpha, Jz)
    basis = list(itertools.combinations(range(N), b))
    index = {c: k for k, c in enumerate(basis)}
    H = np.zeros((len(basis), len(basis)))
    for k, c in enumerate(basis):
        H[k, k] = config_energy(list(c), V, W)
        occ = set(c)
        for i in c:
            j = i + 1                                 # each bond once; add both directions
            if j < N and j not in occ:
                new = tuple(sorted((occ - {i}) | {j}))
                H[index[new], k] += Jx
                H[k, index[new]] += Jx
    return H, index

def even_projector(N, index):
    """
    Dense isometry P (D x D_even) onto the R-even subspace. Columns are
        |c>                        if c is its own mirror
        (|c> + |R c>)/sqrt(2)      otherwise

    Returns:
        P     : (D, D_even) matrix with orthonormal columns
        orbit : dict, configuration c -> column of the orbit {c, R c}
    """
    D = len(index)
    cols, orbit = [], {}
    for c in index:                                    # dict keeps basis order
        if c in orbit:
            continue
        m = tuple(sorted(N - 1 - i for i in c))        # mirror image of c
        v = np.zeros(D)
        if m == c:
            v[index[c]] = 1.0
        else:
            v[index[c]] = v[index[m]] = 1 / np.sqrt(2)
        orbit[c] = orbit[m] = len(cols)
        cols.append(v)
    return np.column_stack(cols), orbit


def leakage_ED(N, b, alpha, Jz, Jx, tol=1e-9):
    """
    Exact time-averaged leakage of every valid codeword in sector b:

        L_ED(S) = 1 - sum_E <v_S|P_E|v_S>^2,    |v_S> = (|S> + |S^c>)/sqrt(2)

    |v_S> is R-even, so only R-even eigenstates contribute.
    |v_S> is exactly the basis vector of the orbit {S, S^c}, so <n|v_S> = U[orbit, n].
    Levels closer than tol * max|E| are grouped into one eigenspace P_E; for a
    nondegenerate spectrum this reduces to the IPR, sum_n <n|v_S>^4.

    One diagonalization serves all codewords of all pair assignments in the sector.

    Returns:
        list of dicts with t, S, occS, occSc, L_ED
    """
    m = N // 2
    if N % 2 or b % 2 or b < 4 or b > m:
        raise ValueError(f"no valid codewords for N = {N}, b = {b}: need N even, b even, 4 <= b <= N/2")
    cw = list(all_codewords(N, b))

    H, index = build_Heff_sector(N, b, alpha, Jz, Jx)  # dense, D x D
    P, orbit = even_projector(N, index)                # dense, D x D_even
    He = P.T @ H @ P                                   # R-even block
    e, U = np.linalg.eigh(He)                          # e ascending, U[:, n] = |n>

    # start index of each (near-)degenerate block of levels
    starts = np.r_[0, np.flatnonzero(np.diff(e) > tol * max(1.0, np.abs(e).max())) + 1]

    k = np.array([orbit[tuple(occS)] for _, _, occS, _ in cw])
    weight_E = np.add.reduceat(U[k, :] ** 2, starts, axis=1)   # <v_S|P_E|v_S> per level E
    P_return = (weight_E ** 2).sum(axis=1)

    return [dict(t="".join(map(str, t)), S=tuple(sorted(S)), occS=occS, occSc=occSc, L_ED=1 - Pr)
            for (t, S, occS, occSc), Pr in zip(cw, P_return)]

def find_best_pair(rows):
    """Best logical pair among all codewords in rows: returns (pair, L_pair)."""
    groups = {}
    for r in rows:
        t = r["t"]
        if t not in groups:
            groups[t] = []
        groups[t].append(r)

    pair_out = None
    L_out = None
    for t in groups:
        words = groups[t]
        words.sort(key=lambda r: r["L_ED"])
        L_pair = words[1]["L_ED"]                     # worse of the two best codewords
        if L_out is None or L_pair < L_out:
            L_out = L_pair
            pair_out = sorted([tuple(words[0]["occS"]), tuple(words[1]["occS"])])
    return tuple(pair_out), L_out

def one_indexed(word):
    """Occupied sites 0..N-1 -> 1..N, for display only."""
    sites = []
    for i in word:
        sites.append(i + 1)
    return sites

In [ ]:
import os
import numpy as np
import qutip as qt

# ---------------- parameters (explicit, not inherited from other cells) ----------------
N, b, alpha = 10, 4, 0.4
Jz, Jx, Bx = 1.0, 0.01, 0.01                   # Jz/Jx = 100
gamma = 1e-3                                   # rate of each single-site jump
times = np.concatenate(([0.0], np.logspace(0, 3, 60)))   # gamma*t from 1e-3 to 1

folder = "Open"
os.makedirs(folder, exist_ok=True)
tag = f"N{N}_b{b}_alpha{alpha}_JzJx{Jz/Jx:g}_gamma{gamma:g}"

# ---------------- step 1: optimal logical pair at N = 10 ----------------
rows = leakage_ED(N, b, alpha, Jz, Jx)
pair, L_pair = find_best_pair(rows)
occ0, occ1 = pair
print("0_L sites:", one_indexed(occ0), " 1_L sites:", one_indexed(occ1), f" L_pair = {L_pair:.2e}")

# ---------------- step 2: full spin Hamiltonian, Eq. (HAM) ----------------
def site_op(op, i):
    ops = [qt.qeye(2)] * N
    ops[i] = op
    return qt.tensor(ops)

sx = [site_op(qt.sigmax(), i) for i in range(N)]
sy = [site_op(qt.sigmay(), i) for i in range(N)]
sz = [site_op(qt.sigmaz(), i) for i in range(N)]

H = 0
for i in range(N):
    for j in range(i + 1, N):
        H += Jz / abs(i - j) ** alpha * sz[i] * sz[j]
for i in range(N):
    H += Bx * sx[i]
for i in range(N - 1):
    H += Jx * sx[i] * sx[i + 1]

# ---------------- step 3: logical states ----------------
def config_ket(occ):
    """Product state with an excitation (basis(2,1)) on each site in occ."""
    states = []
    for i in range(N):
        if i in occ:
            states.append(qt.basis(2, 1))
        else:
            states.append(qt.basis(2, 0))
    return qt.tensor(states)

def codeword(occ):
    mirror = sorted(N - 1 - i for i in occ)
    return (config_ket(occ) + config_ket(mirror)).unit()

zero_L = codeword(occ0)
one_L = codeword(occ1)
plus_L = (zero_L + one_L).unit()

# ---------------- step 4: projector onto states that pass every check ----------------
# checks: sector b, pair totals of A, M = 0, and reflection R = +1
A = sorted(min(i, N - 1 - i) for i in occ0)    # active pairs
dim = 2 ** N
D = np.zeros(dim)                              # diagonal checks
R = np.zeros((dim, dim))                       # reflection operator
for k in range(dim):
    bits = format(k, f"0{N}b")                 # site 0 is the leftmost bit
    R[int(bits[::-1], 2), k] = 1.0

    occ = [i for i in range(N) if bits[i] == "1"]
    ok = len(occ) == b
    n_left = 0
    for p in range(N // 2):
        total = (bits[p] == "1") + (bits[N - 1 - p] == "1")
        if p in A:
            ok = ok and total == 1
            n_left += bits[p] == "1"
        else:
            ok = ok and total == 0
    ok = ok and n_left == len(A) // 2          # M = 0
    if ok:
        D[k] = 1.0

dims = [[2] * N, [2] * N]
R = qt.Qobj(R, dims=dims)
Pi = 0.5 * (qt.qeye([2] * N) + R) * qt.Qobj(np.diag(D), dims=dims)

# ---------------- step 5: the three noise models, Eq. (lindblad) ----------------
local_xyz = []
for i in range(N):
    local_xyz += [np.sqrt(gamma) * sx[i], np.sqrt(gamma) * sy[i], np.sqrt(gamma) * sz[i]]

local_z = []
for i in range(N):
    local_z.append(np.sqrt(gamma) * sz[i])

total_z = 0
for i in range(N):
    total_z += sz[i]
collective_z = [np.sqrt(gamma) * total_z]

# (label for plots, short name for the file, jump operators)
noise_models = [("local X, Y, Z", "localXYZ", local_xyz),
                ("local Z", "localZ", local_z),
                ("collective Z", "collectiveZ", collective_z)]

# ---------------- step 6: evolve, then save one file per noise model ----------------
e_ops = [Pi, one_L * one_L.dag(), zero_L * one_L.dag()]

for label, short, c_ops in noise_models:
    filename = os.path.join(folder, f"{tag}_{short}.txt")
    if os.path.exists(filename):
        print("already saved, skipping:", filename)
        continue

    print("running:", label)
    r0 = qt.mesolve(H, zero_L * zero_L.dag(), times, c_ops=c_ops, e_ops=e_ops)
    rP = qt.mesolve(H, plus_L * plus_L.dag(), times, c_ops=c_ops, e_ops=e_ops)

    p_pass0 = np.real(r0.expect[0])
    p_passP = np.real(rP.expect[0])
    detected = 1 - p_pass0                                      # checks failed
    flip = np.real(r0.expect[1]) / p_pass0                      # undetected bit flip from |0_L>
    dephase = 1 - 2 * np.abs(rP.expect[2]) / p_passP            # undetected loss of coherence from |+_L>

    header = (f"noise={label}\n"
              f"N={N} b={b} alpha={alpha} Jz={Jz} Jx={Jx} Bx={Bx} gamma={gamma}\n"
              f"0_L={' '.join(str(i) for i in occ0)} (0-indexed)\n"
              f"1_L={' '.join(str(i) for i in occ1)} (0-indexed)\n"
              f"L_pair={L_pair:.6e}\n"
              f"columns: t  detected  undetected_flip  undetected_dephasing")
    data = np.column_stack([times, detected, flip, dephase])
    np.savetxt(filename, data, header=header, fmt="%.10e")
    print("saved:", filename)

0_L sites: [1, 2, 6, 7]  1_L sites: [1, 5, 7, 9]  L_pair = 8.38e-03
running: local X, Y, Z


KeyboardInterrupt: 